<a href="https://colab.research.google.com/github/Harman8815/AMD-Hackathon/blob/main/amd_hackathon_v_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AMD Hackathon: Vector-based Knowledge Retrieval

This notebook implements the first two phases of our RAG (Retrieval-Augmented Generation) pipeline:
1. **Setup**: Environment configuration and model initialization.
2. **Indexing**: Parsing PDF documents, chunking text, and storing embeddings in ChromaDB.

## Section 1: Foundation & Infrastructure
This section focuses on preparing the environment and initializing core models. We utilize `SentenceTransformer` for vector embeddings and `ChromaDB` for high-performance persistent storage.

**Git Status**: Environment initialized and verified.
**Commit**: `feat: initialize RAG environment with ChromaDB and SentenceTransformers`

## Step 1: Environment Setup
In this step, we install the necessary libraries and initialize our embedding model and vector database client.

In [1]:
!pip install chromadb langchain langchain-community sentence-transformers pymupdf

In [2]:
import fitz  # PyMuPDF
import chromadb
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize Embedding Model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize ChromaDB Client
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="design_guides")

print("Setup Complete: ChromaDB and Model initialized.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Setup Complete: ChromaDB and Model initialized.


## Section 2: Intelligent Document Parsing
Here we implement a robust PDF extraction pipeline using `PyMuPDF`. The text is split into contextual chunks using `RecursiveCharacterTextSplitter` to ensure that relevance is maintained during the retrieval phase.

**Git Status**: Document parsing and chunking logic implemented.
**Commit**: `feat: implement PDF text extraction and recursive chunking pipeline`

## Step 2: Create Chunks and Store in Database
This section handles the document processing pipeline: extraction, splitting, and vector storage.

In [5]:
def extract_pdf_text(pdf_path):
    """Extracts all text from a PDF file."""
    text = ""
    doc = fitz.open(pdf_path)
    for page in tqdm(doc, desc=f"Reading {pdf_path}"):
        text += page.get_text()
    return text

# Configure Text Splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", " ", ""]
)

In [7]:
# List of documents to process
pdfs = [
    "guide 1.pdf",
    "guide 2.pdf",
    "guide 3.pdf"
]

all_chunks = []

for pdf in pdfs:
    print(f"\nProcessing {pdf}")
    text = extract_pdf_text(pdf)
    chunks = splitter.split_text(text)

    print(f"Chunks Created: {len(chunks)}")

    for chunk in chunks:
        if len(chunk.strip()) > 100:
            all_chunks.append({
                "source": pdf,
                "content": chunk
            })


Processing guide 1.pdf


Reading guide 1.pdf:   0%|          | 0/39 [00:00<?, ?it/s]

Chunks Created: 32

Processing guide 2.pdf


Reading guide 2.pdf:   0%|          | 0/34 [00:00<?, ?it/s]

Chunks Created: 82

Processing guide 3.pdf


Reading guide 3.pdf:   0%|          | 0/86 [00:00<?, ?it/s]

Chunks Created: 308


In [8]:
# Batch Upload to ChromaDB
batch_size = 32

for i in tqdm(range(0, len(all_chunks), batch_size), desc="Uploading to Chroma"):
    batch = all_chunks[i:i + batch_size]
    texts = [x["content"] for x in batch]
    embeddings = model.encode(texts, show_progress_bar=False)

    collection.add(
        ids=[f"chunk_{i+j}" for j in range(len(batch))],
        documents=texts,
        embeddings=embeddings.tolist(),
        metadatas=[{"source": x["source"]} for x in batch]
    )

print(f"\nSuccessfully stored {collection.count()} chunks in the database.")

Uploading to Chroma:   0%|          | 0/14 [00:00<?, ?it/s]


Successfully stored 422 chunks in the database.


## Step 3: Verification & Testing
Let's verify the storage by performing a sample similarity search.

In [9]:
# Sample Search
query = "minimum wall thickness"
query_embedding = model.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

print("=" * 50)
print(f"Search Results for: '{query}'")
print("=" * 50)

for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1} (Source: {results['metadatas'][0][i]['source']})")
    print("-" * 20)
    print(doc[:500] + "...")

Search Results for: 'minimum wall thickness'

Result 1 (Source: guide 3.pdf)
--------------------
Indicate radii at alinsile and outside corners to the 
maxinum which a design will albw. 
Draft Angles  
Draft is necessary for the ejection of the parts from the
mold.  Always design with draft angles.  Recommended
draft angle is normally 1° with 1/2° on ribs.  Some draft
angle is better than none and more draft is desirable if the
design permits.  Where minimum draft is desired, good
polishing is recommended and feature depth should not
exceed .5in.
Wall Thickness
The number one rule for desig...

Result 2 (Source: guide 1.pdf)
--------------------
no more than 60% of the overall wall thickness. 
•	
Walls: Consider how bosses will attach to walls — they 
need to align properly. 
TYPICAL BOSS 
DESIGN 
CONNECTING BOSSES TO WALL
BOSS IN ATTACHMENT WALL
Open bosses maintain uniform 
thickness in the attachment wall
Injection Molding Design Guide  |  19
STEP 1: Start Your Design
Tolerances
S

In [10]:
# Knowledge Base Summary
print("=" * 50)
print("Knowledge Base Summary")
print("=" * 50)
print("PDFs processed:", len(pdfs))
print("Total chunks created:", len(all_chunks))
print("Stored in DB:", collection.count())

Knowledge Base Summary
PDFs processed: 3
Total chunks created: 422
Stored in DB: 422


## Section 3: Semantic Rule Extraction
By leveraging LLMs (Gemini), we extract structured engineering rules from the retrieved text chunks. This process transforms unstructured PDF data into a machine-readable JSON rule-book.

**Git Status**: Rule retrieval and LLM extraction pipeline active.
**Commit**: `feat: implement LLM-based engineering rule extraction from vector store`

In [ ]:
queries = [
    "minimum requirement",
    "maximum limit",
    "shall be",
    "must be",
    "recommended value",
    "design requirement"
]

rule_chunks = []

for query in queries:
    results = collection.query(
        query_texts=[query],
        n_results=10
    )

    for doc in results["documents"][0]:
        rule_chunks.append(doc)

rule_chunks = list(set(rule_chunks))

print(f"Retrieved {len(rule_chunks)} unique rule chunks")

Retrieved 46 unique rule chunks


In [11]:
!pip install -q google-genai

In [12]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(
    api_key=api_key
)

print("Connected")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hello"
)

print(response.text)

Hello! How can I help you today?


In [ ]:
import json
from tqdm.auto import tqdm

all_rules = []

for chunk in tqdm(rule_chunks):

    prompt = f"""
You are an engineering standards analyst.

Extract all explicit engineering requirements, constraints, limits, minimums, maximums and recommendations from the text.

Return ONLY a valid JSON array.

Format:

[
  {{
    "parameter": "",
    "operator": "",
    "value": "",
    "unit": "",
    "rule_text": ""
  }}
]

Allowed operators:
>, >=, <, <=, ==

If no rules are found return:

[]

Text:
{chunk}
"""

    try:

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        result = response.text.strip()

        if result.startswith("```json"):
            result = result.replace("```json", "").replace("```", "").strip()

        elif result.startswith("```"):
            result = result.replace("```", "").strip()

        rules = json.loads(result)

        if isinstance(rules, list):
            all_rules.extend(rules)

    except Exception as e:
        print(f"Error: {e}")

print(f"Extracted {len(all_rules)} rules")

  0%|          | 0/46 [00:00<?, ?it/s]

Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 18.508652324s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'qu

In [ ]:
unique_rules = {}

for rule in all_rules:

    key = (
        rule.get("parameter"),
        rule.get("operator"),
        str(rule.get("value"))
    )

    unique_rules[key] = rule

all_rules = list(unique_rules.values())

print(f"Unique Rules: {len(all_rules)}")

In [ ]:
with open("rules.json", "w") as f:
    json.dump(all_rules, f, indent=2)

print("rules.json saved")

rules.json saved


## Section 4: Automated Design Validation
The final stage involves matching user-provided design attributes against our extracted rule-book. We use cosine similarity to find the most relevant parameters and apply mathematical operators to determine compliance.

**Git Status**: Validation engine and compliance reporting completed.
**Commit**: `feat: implement semantic parameter matching and compliance validation engine`

In [13]:
import json
import operator
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

OPS = {
    ">": operator.gt,
    ">=": operator.ge,
    "<": operator.lt,
    "<=": operator.le,
    "==": operator.eq
}


def validate_design(design, rules_file="rules.json"):

    with open(rules_file, "r") as f:
        rules = json.load(f)

    results = []

    attributes = design["attributes"]

    attribute_names = list(attributes.keys())

    attribute_embeddings = embedding_model.encode(
        attribute_names
    )

    for rule in rules:

        try:

            if not all(
                k in rule
                for k in [
                    "parameter",
                    "operator",
                    "value"
                ]
            ):
                continue

            rule_parameter = rule["parameter"]

            rule_embedding = embedding_model.encode(
                [rule_parameter]
            )

            similarities = cosine_similarity(
                rule_embedding,
                attribute_embeddings
            )[0]

            best_idx = np.argmax(similarities)

            best_attribute = attribute_names[
                best_idx
            ]

            similarity = similarities[
                best_idx
            ]

            if similarity < 0.60:
                continue

            actual = float(
                attributes[
                    best_attribute
                ]
            )

            expected = float(
                rule["value"]
            )

            operator_symbol = rule[
                "operator"
            ]

            if operator_symbol not in OPS:
                continue

            passed = OPS[
                operator_symbol
            ](
                actual,
                expected
            )

            results.append({
                "rule_parameter":
                    rule_parameter,
                "matched_attribute":
                    best_attribute,
                "similarity":
                    round(
                        similarity,
                        3
                    ),
                "actual":
                    actual,
                "expected":
                    expected,
                "operator":
                    operator_symbol,
                "status":
                    (
                        "PASS"
                        if passed
                        else "FAIL"
                    ),
                "rule_text":
                    rule.get(
                        "rule_text",
                        ""
                    )
            })

        except:
            continue

    return results

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
design = {
    "attributes": {
        "part_size": 1.5,
        "overall_length": 1.2,
        "diameter": 0.75
    }
}

results = validate_design(design)

for result in results:
    print(result)

{'rule_parameter': 'radius', 'matched_attribute': 'diameter', 'similarity': np.float32(0.654), 'actual': 0.75, 'expected': 0.02, 'operator': '==', 'status': 'FAIL', 'rule_text': '.020in R'}
{'rule_parameter': 'Part Dimension', 'matched_attribute': 'part_size', 'similarity': np.float32(0.679), 'actual': 1.5, 'expected': 1.75, 'operator': '<', 'status': 'PASS', 'rule_text': '<1.75'}
{'rule_parameter': 'Part Dimension', 'matched_attribute': 'part_size', 'similarity': np.float32(0.679), 'actual': 1.5, 'expected': 0.75, 'operator': '>=', 'status': 'PASS', 'rule_text': '0.75 to 1.50'}
{'rule_parameter': 'Part Dimension', 'matched_attribute': 'part_size', 'similarity': np.float32(0.679), 'actual': 1.5, 'expected': 1.5, 'operator': '<=', 'status': 'PASS', 'rule_text': '0.75 to 1.50'}
{'rule_parameter': 'Part Dimension', 'matched_attribute': 'part_size', 'similarity': np.float32(0.679), 'actual': 1.5, 'expected': 1.5, 'operator': '>', 'status': 'FAIL', 'rule_text': '>1.50'}
{'rule_parameter': '

In [16]:
passes = sum(
    1 for r in results
    if r["status"] == "PASS"
)

fails = sum(
    1 for r in results
    if r["status"] == "FAIL"
)

print("=" * 50)
print("VALIDATION SUMMARY")
print("=" * 50)
print(f"PASS : {passes}")
print(f"FAIL : {fails}")
print(f"TOTAL: {len(results)}")

VALIDATION SUMMARY
PASS : 3
FAIL : 3
TOTAL: 6


In [17]:
passed_rules = []
failed_rules = []

for item in results:

    rule_text = item.get(
        "rule_text",
        ""
    )

    if not rule_text:
        continue

    if item["status"] == "FAIL":
        failed_rules.append(rule_text)
    else:
        passed_rules.append(rule_text)

context = f"""
FAILED RULES

{chr(10).join(failed_rules)}

PASSED RULES

{chr(10).join(passed_rules)}
"""

print(context)


FAILED RULES

.020in R
>1.50
The critical dimension, and the one in question, is 2.002 in. in diameter.

PASSED RULES

<1.75
0.75 to 1.50
0.75 to 1.50



In [18]:
user_query = """
Is this design compliant?
What are the major risks?
Suggest corrective actions.
"""

In [19]:
import requests

OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen3:4b"

prompt = f"""
You are an expert design compliance engineer.

Analyze the design using the rule validation results below.

{context}

User Question:
{user_query}

Instructions:

1. Explain overall compliance status.
2. Highlight failed rules.
3. Explain risks.
4. Suggest corrective actions.
5. Reference the relevant rules.
"""

response = requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False
    }
)

answer = response.json()["response"]

print(answer)

### Expert Design Compliance Analysis Report  

**1. Overall Compliance Status**  
❌ **Non-compliant** (Critical failure detected). The design fails **one critical rule** related to a radius dimension. While all other rules passed, the failure of this key rule poses significant engineering risks that must be addressed before the design can be approved for manufacturing or use.  

---

### 2. Highlighted Failed Rules  
**Failed Rule**:  
`Radius (R) > 1.50`  
**Actual Value**: `0.020 in` (This is a critical misinterpretation—see *Section 3* for clarification).  
**Critical Dimension**: `2.002 in. diameter` (implying a **radius of 1.001 in.**, *not* 0.020 in.).  

**Why this is a critical failure**:  
The rule states that the **radius must exceed 1.50 inches** (i.e., `R > 1.50`). However, the *actual* radius for the critical dimension (2.002 in. diameter) is **1.001 in.** (not 0.020 in.). The reported value of `.020in R` is likely a **typo** in the rule output (e.g., missing a decimal po

In [ ]:
response = requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={
        "model": "qwen3:4b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": 500
        }
    }
)